# Week 7: Retrieval-Augmented Generation (RAG)

**Applied Generative AI**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

---

## Learning Objectives

By the end of this notebook, you will be able to:

- **Understand RAG architecture** — how retrieval and generation work together
- **Implement embedding-based retrieval** — SentenceTransformers, FAISS, and ChromaDB
- **Build production RAG pipelines** — from chunking to generation
- **Compare chunking strategies** — fixed-size, sentence-based, recursive splitting
- **Evaluate RAG quality** — identify when RAG helps and when it fails

---
## ⚙️ Setup — Install & Import

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers datasets chromadb langchain langchain-community

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain_community.llms import HuggingFacePipeline

---
## 1. Why RAG?

**Retrieval-Augmented Generation (RAG)** combines two components:
- **Retriever**: Finds relevant context/documents from a knowledge base
- **Generator**: Uses an LLM to answer queries, augmented with retrieved context

### When to Use RAG vs Fine-Tuning vs Prompting

| Approach | Best For | Pros | Cons |
|----------|----------|------|------|
| **Prompting** | General tasks, quick prototyping | No training, instant | Limited by context, brittle |
| **RAG** | Knowledge-heavy, frequently updated data | No retraining for new docs, traceable sources | Retrieval quality limits output |
| **Fine-Tuning** | Domain-specific behavior, style | Learns new patterns | Needs data, compute, risk of forgetting |

**Use RAG when:**
- You have a corpus of documents that changes frequently
- You need traceable, grounded answers
- Fine-tuning is too expensive or data is scarce

### RAG Architecture (Text Diagram)

```
┌─────────────┐     ┌──────────────────┐     ┌─────────────────┐
│   User      │────▶│    Retriever      │────▶│   Knowledge     │
│   Query     │     │  (Embed + Search) │     │   Base (FAISS)  │
└─────────────┘     └────────┬─────────┘     └─────────────────┘
                             │
                             │ Retrieved Context
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│   Generator (LLM)                                                │
│   Prompt: "Context: {retrieved}\\n\nQuestion: {query}\\nAnswer:"   │
└─────────────────────────────────────────────────────────────────┘
                             │
                             ▼
┌─────────────────┐
│   Final Answer  │
└─────────────────┘
```

---
## 2. Building a Knowledge Base

A knowledge base is a collection of text documents. Before indexing, we need to **chunk** them into smaller pieces for retrieval.

### 2.1 Sample Documents

In [ ]:
# Sample knowledge base (from Module3_RAG)
knowledge_base = [
    "The Eiffel Tower is located in Paris, France.",
    "The Great Wall of China is visible from space only with aid.",
    "Python is a popular programming language for AI and machine learning.",
    "The human brain contains around 86 billion neurons.",
    "Photosynthesis in plants converts sunlight into chemical energy."
]

print("Knowledge Base:")
for i, doc in enumerate(knowledge_base):
    print(f"[{i}] {doc}")

### 2.2 Chunking Strategies

**Chunking** splits long documents into smaller units for retrieval. Different strategies affect retrieval quality:

- **Fixed-size**: Simple, predictable chunks
- **Sentence-based**: Respects natural boundaries
- **Recursive character splitting**: Tries separators (\n\n, \n, space) in order

In [ ]:
# Long document for chunking demo
long_doc = """
The Eiffel Tower is located in Paris, France. It was built in 1889 for the World's Fair.

The Eiffel Tower stands 330 meters tall. It is one of the most visited monuments in the world.

Millions of tourists visit the Eiffel Tower every year. The structure was designed by Gustave Eiffel.
"""

# Fixed-size chunking
fixed_splitter = CharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separator=" "
)
fixed_chunks = fixed_splitter.split_text(long_doc)

print("=== Fixed-Size Chunking (chunk_size=100, overlap=20) ===")
for i, c in enumerate(fixed_chunks):
    print(f"Chunk {i}: {c[:80]}..." if len(c) > 80 else f"Chunk {i}: {c}")

In [ ]:
# Recursive character splitting (recommended for production)
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""]
)
recursive_chunks = recursive_splitter.split_text(long_doc)

print("=== Recursive Character Splitting (chunk_size=150, overlap=30) ===")
for i, c in enumerate(recursive_chunks):
    print(f"Chunk {i}: {c}")

In [ ]:
# Compare chunk sizes
print("Fixed-size chunks:", len(fixed_chunks), "chunks")
print("Recursive chunks:", len(recursive_chunks), "chunks")
print("\nAverage chunk length (fixed):", np.mean([len(c) for c in fixed_chunks]))
print("Average chunk length (recursive):", np.mean([len(c) for c in recursive_chunks]))

---
## 3. Embeddings and Vector Search

We convert text into **embeddings** (dense vectors) so we can search using vector similarity.

### 3.1 Generate Embeddings with SentenceTransformers

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(knowledge_base)

print(f"Embedding shape: {kb_embeddings.shape}")
print(f"Embedding dimension: {kb_embeddings.shape[1]}")

### 3.2 Build FAISS Index

In [ ]:
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print(f"Knowledge base indexed with {index.ntotal} documents.")

### 3.3 Similarity Search and Retrieval

In [ ]:
def retrieve(query, top_k=2):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    results = [knowledge_base[i] for i in indices[0]]
    return results, distances[0]

query = "Where is the Eiffel Tower?"
print("Query:", query)
docs, scores = retrieve(query)
print("Retrieved Docs:")
for d, s in zip(docs, scores):
    print(f"  - {d} (L2 distance: {s:.2f})")

### 3.4 ChromaDB as Alternative Vector Store

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

chroma_db = Chroma.from_texts(
    texts=knowledge_base,
    embedding=embeddings,
    collection_name="demo_kb"
)

chroma_results = chroma_db.similarity_search_with_score("Where is the Eiffel Tower?", k=2)
print("ChromaDB retrieval:")
for doc, score in chroma_results:
    print(f"  - {doc.page_content} (score: {score:.2f})")

---
## 4. The RAG Pipeline

**Retriever + Generator** architecture: retrieve relevant docs, then feed them as context to the LLM.

### 4.1 End-to-End RAG with Flan-T5

In [ ]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")

def rag(query, top_k=3):
    retrieved_docs = retrieve(query, top_k=top_k)[0]
    context = "\n".join(retrieved_docs)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    answer = generator(prompt, max_length=200)[0]['generated_text']
    return answer.strip(), retrieved_docs

In [ ]:
query = "What city is the Eiffel Tower in?"
answer, docs = rag(query)
print("Query:", query)
print("Retrieved Docs:", docs)
print("RAG Answer:", answer)

### 4.2 Compare RAG vs Direct LLM Output

In [ ]:
query = "What city is the Eiffel Tower in?"

print("=== Direct Query (No RAG) ===")
direct_answer = generator(query, max_length=50)[0]['generated_text']
print(direct_answer)

print("\n=== With RAG ===")
answer, docs = rag(query)
print("Retrieved Docs:", docs)
print("RAG Answer:", answer)

In [ ]:
# Larger knowledge base: AG News dataset
from datasets import load_dataset

ag_news = load_dataset("ag_news", split="train[:500]")
kb_large = [x["text"] for x in ag_news]

kb_embeddings_large = model.encode(kb_large, show_progress_bar=True)
index_large = faiss.IndexFlatL2(kb_embeddings_large.shape[1])
index_large.add(kb_embeddings_large)

def retrieve_large(query, top_k=3):
    q_emb = model.encode([query])
    _, indices = index_large.search(q_emb, top_k)
    return [kb_large[i] for i in indices[0]]

def rag_large(query, top_k=3):
    docs = retrieve_large(query, top_k=top_k)
    context = "\n".join(docs)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    answer = generator(prompt, max_length=200)[0]['generated_text']
    return answer.strip(), docs

query = "What happened in the stock market today?"
print("=== Direct Query (No RAG) ===")
print(generator(query, max_length=80)[0]['generated_text'])
print("\n=== With RAG ===")
answer, docs = rag_large(query)
print("Retrieved Docs:")
for d in docs:
    print(f"  - {d[:120]}...")
print("\nRAG Answer:", answer)

---
## 5. RAG with LangChain

LangChain abstracts the retriever, vector store, and LLM into a unified pipeline.

In [ ]:
llm = HuggingFacePipeline(pipeline=generator)

vectorstore = Chroma.from_texts(
    texts=knowledge_base,
    embedding=embeddings,
    collection_name="langchain_rag"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

print("LangChain RAG pipeline ready.")

In [ ]:
result = qa_chain.invoke({"query": "Where is the Eiffel Tower?"})
print("Query: Where is the Eiffel Tower?")
print("LangChain RAG Answer:", result["result"])

---
## 6. When RAG Fails

RAG is not a silver bullet. Failures occur in two places:

### Retrieval Failures
- **Wrong chunks**: Query retrieves irrelevant documents
- **Missing information**: The knowledge base doesn't contain the answer
- **Irrelevant results**: Embedding similarity doesn't match semantic relevance

### Generation Failures
- **Model ignores context**: LLM relies on its own knowledge instead of retrieved docs
- **Hallucination despite context**: Model generates plausible but incorrect answers

### Demonstrate a Failure Case

In [ ]:
# Query about something NOT in our knowledge base
query = "Where is the Statue of Liberty?"

docs, scores = retrieve(query, top_k=2)
print("Query:", query)
print("Retrieved Docs (may be irrelevant):")
for d, s in zip(docs, scores):
    print(f"  - {d} (L2: {s:.2f})")

answer, _ = rag(query)
print("\nRAG Answer:", answer)
print("\n⚠️ The retrieval returned Eiffel Tower docs (wrong topic).")
print("RAG can fail when: (1) KB lacks the answer, (2) retrieval returns wrong chunks.")

---
## 7. Exercises

### Exercise 1: Build a RAG System on a Custom Knowledge Base

Create a RAG system with 5–10 documents of your choice (e.g., course syllabus, Wikipedia excerpts, or a short article).

1. Define your documents
2. Chunk them (if long)
3. Build embeddings and FAISS index
4. Implement retrieve and rag functions
5. Test with 3–5 queries

In [ ]:
# Your custom knowledge base here
# custom_docs = ["...", "...", ...]
# ... implement RAG pipeline ...

### Exercise 2: Compare Chunking Strategies

Take a longer document (e.g., a Wikipedia article).

1. Chunk it with fixed-size (chunk_size=100) vs recursive (chunk_size=100)
2. Build FAISS indices for both
3. Run the same query 5 times and compare retrieval quality
4. Which strategy retrieves more relevant chunks?

In [ ]:
# Compare fixed vs recursive chunking on retrieval quality
# ... implement comparison ...

### Exercise 3: Find a RAG Failure and Diagnose

1. Find a query where RAG fails (wrong answer or irrelevant retrieval)
2. Diagnose: Is it a retrieval failure or a generation failure?
3. Propose one fix (e.g., better chunking, more retrieval, different prompt)

In [ ]:
# Find a failure case and diagnose
# query = "..."
# ... run RAG, analyze, propose fix ...

---
## 8. Responsible AI Reflection

RAG systems can give the impression of authoritative, well-sourced answers while quietly hallucinating or retrieving irrelevant context. Users trust RAG more because it appears to cite sources — but that trust may be misplaced.

**Reflection questions:**
- When is RAG more dangerous than a simple LLM?
- What attribution or transparency mechanisms should a production RAG system include?
- How can we surface retrieval confidence to users?

---
## 9. Summary & Next Steps

**Summary:**
- RAG = Retriever + Generator; use when knowledge is dynamic and traceability matters
- Chunking: fixed-size, sentence-based, recursive — choose based on document structure
- Embeddings: SentenceTransformers; Vector stores: FAISS, ChromaDB
- LangChain abstracts retrieval and generation into reusable pipelines
- RAG fails when retrieval is wrong or the model ignores context

**Next Steps:**
- Week 8: Agentic workflows; RAG as a tool for agents
- Advanced: Hybrid search (dense + sparse), reranking, multi-hop retrieval